# 02 — Orthographic Globe Plot from Scratch

Build a publication-quality dark-themed orthographic globe step by step — **no `noaawc`**, just cartopy + matplotlib.

---
**Stack used:** `noawclg`, `xarray`, `numpy`, `matplotlib`, `cartopy`, `cmocean`

In [ ]:
import noawclg
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean

## 1. Apply the dark theme

Set matplotlib rcParams globally so every figure uses the same dark palette.

In [ ]:
BG      = "#0d1117"   # figure / axes background
LAND    = "#13171d"   # land fill
OCEAN   = "#0d1520"   # ocean fill
COAST   = "#2d6db5"   # coastline colour
BORDER  = "#2a2f36"   # country borders
TEXT    = "#e6edf3"   # titles, labels
SUBTEXT = "#8b949e"   # secondary text / tick labels
BOXBG   = "#161b22"   # info-box background
BOXEDGE = "#30363d"   # info-box border

plt.rcParams.update({
    "figure.facecolor":  BG,
    "axes.facecolor":    BG,
    "savefig.facecolor": BG,
    "text.color":        TEXT,
    "axes.labelcolor":   TEXT,
    "xtick.color":       SUBTEXT,
    "ytick.color":       SUBTEXT,
    "font.family":       "monospace",
})

## 2. Load GFS data

In [ ]:
ds = noawclg.load(
    var="t2m",
    run_date="20260418",
    cycle="00",
    forecast_hours=[0],
)

lat = ds["lat"].values
lon = ds["lon"].values

# K → °C, first time step
field = ds["t2m"].isel(time=0).values - 273.15

# Convert longitude 0–360 → −180/180
lon_180 = np.where(lon > 180, lon - 360, lon)
order   = np.argsort(lon_180)
lon_180 = lon_180[order]
field   = field[:, order]

## 3. Define colormap and levels

`BoundaryNorm` maps data values to discrete colormap indices — gives a crisp,
banded look instead of continuous shading.

In [ ]:
import copy

levels = np.arange(-20, 52, 2)          # −20 … 50 °C, step 2
cmap   = copy.copy(cmocean.cm.thermal)  # copy so we can mutate it
cmap.set_under(alpha=0)                  # transparent for out-of-range below
cmap.set_bad(alpha=0)                    # transparent for NaN
norm   = mcolors.BoundaryNorm(levels, ncolors=cmap.N, clip=False)

## 4. Create the Orthographic axes

`ccrs.Orthographic(central_longitude, central_latitude)` places the camera directly above that point.

In [ ]:
# Camera position — try different values
CENTRAL_LON = -50.0
CENTRAL_LAT = -15.0

proj = ccrs.Orthographic(
    central_longitude=CENTRAL_LON,
    central_latitude=CENTRAL_LAT,
)

fig = plt.figure(figsize=(10, 10), facecolor=BG)
ax  = fig.add_subplot(1, 1, 1, projection=proj)
ax.set_facecolor(BG)
ax.set_global()   # show the full hemisphere

## 5. Add geographic features

In [ ]:
ax.add_feature(cfeature.OCEAN.with_scale("110m"),
               facecolor=OCEAN, zorder=0)
ax.add_feature(cfeature.LAND.with_scale("110m"),
               facecolor=LAND,  zorder=0)
ax.add_feature(cfeature.COASTLINE.with_scale("110m"),
               edgecolor=COAST, linewidth=0.6, zorder=3)
ax.add_feature(cfeature.BORDERS.with_scale("110m"),
               edgecolor=BORDER, linewidth=0.4, zorder=3)

## 6. Plot the field with pcolormesh

Always pass `transform=ccrs.PlateCarree()` — this tells cartopy that your data
coordinates are in plain lat/lon, and it projects them onto the chosen projection.

In [ ]:
cf = ax.pcolormesh(
    lon_180, lat, field,
    cmap=cmap,
    norm=norm,
    transform=ccrs.PlateCarree(),
    zorder=1,
)

## 7. Add contour lines (optional)

In [ ]:
# Subsample to keep contour fast (every 3rd point)
ax.contour(
    lon_180[::3], lat[::3], field[::3, ::3],
    levels=levels[::5],
    colors="white",
    linewidths=0.25,
    alpha=0.4,
    transform=ccrs.PlateCarree(),
    zorder=2,
)

## 8. Colorbar

In [ ]:
cb = fig.colorbar(cf, ax=ax, orientation="horizontal",
                  pad=0.03, fraction=0.03, shrink=0.85)
cb.set_label("2m Temperature (°C)", fontsize=8, color=SUBTEXT)
cb.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb.outline.set_edgecolor(BOXEDGE)

## 9. Title and info box

In [ ]:
date_str = str(ds["t2m"].time.values[0])[:10]

ax.set_title("2m Temperature", fontsize=10, fontweight="bold",
             color=TEXT, loc="left", pad=6)
ax.set_title(f"GFS 00z  {date_str}", fontsize=7,
             color=SUBTEXT, loc="right", pad=6)

# Info box — top-right corner
ax.text(
    0.995, 0.995,
    f"var: t2m — 2-metre temperature\nDate: {date_str}  cycle: 00z",
    transform=ax.transAxes,
    ha="right", va="top",
    fontsize=7, color="#828283",
    fontfamily="monospace", fontweight="bold",
    linespacing=1.55,
    bbox=dict(boxstyle="round,pad=0.45", facecolor=BOXBG,
              edgecolor=BOXEDGE, linewidth=0.8, alpha=0.88),
    zorder=10,
)

## 10. Save and display

In [ ]:
fig.tight_layout()
fig.savefig("ortho_t2m.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ortho_t2m.png")

## 11. Exercises

1. **Change the camera** — try `CENTRAL_LON=0, CENTRAL_LAT=20` (Europe/Africa view) and `CENTRAL_LON=140, CENTRAL_LAT=35` (Asia/Pacific).
2. **Change the variable** — load `"prmsl"` (divide by 100 for hPa) and update the levels/cmap.
3. **Add a marker** — use `ax.plot(lon, lat, marker="o", transform=ccrs.PlateCarree())` to mark a city.
4. **Animate** — loop over `forecast_hours=range(0, 121, 3)`, call `ax.cla()` + redraw each frame, save with `imageio`.

---
**Next:** [03_flat_regional_map.ipynb](03_flat_regional_map.ipynb)